In [ ]:
import os
import random
import time

import pandas as pd
from dotenv import load_dotenv
from nba_api.stats.endpoints import commonteamroster
from nba_api.stats.static import teams
from sqlalchemy import create_engine, inspect


# helper functions
def normalize_for_postgres(df):
    """Normalize DataFrame column names for PostgreSQL"""
    df = df.copy()
    df.columns = df.columns.str.lower().str.replace(" ", "_")
    return df


load_dotenv()
DATABASE_URL = os.getenv("DATABASE_URL")
engine = create_engine(DATABASE_URL)

# --- NEW: INITIALIZE SEEN_PLAYER_IDS FROM DATABASE ---
print("Checking database for existing players...")
inspector = inspect(engine)
seen_player_ids = set()

if "players" in inspector.get_table_names():
    # Fetch IDs already in the database to avoid duplicates/errors
    existing_ids_df = pd.read_sql("SELECT player_id FROM players", engine)
    seen_player_ids = set(existing_ids_df["player_id"].tolist())
    print(f"Found {len(seen_player_ids)} existing players in the database.")
else:
    print("Table 'players' does not exist yet. Starting fresh.")

# Set target seasons
seasons = ["2024-25"]

nba_teams = teams.get_teams()
print(f"Found {len(nba_teams)} teams in total\n")

all_players = []
team_count = 0

for team in nba_teams:
    team_id = team["id"]
    team_name = team["full_name"]
    team_count += 1

    print(f"[{team_count}/{len(nba_teams)}] Processing team: {team_name}")

    try:
        for season in seasons:
            time.sleep(random.uniform(1.5, 3.0))  # Avoid rate limiting

            try:
                roster = commonteamroster.CommonTeamRoster(team_id=team_id, season=season)
                roster_df = roster.get_data_frames()[0]

                # Filter out players we've already seen (in DB or earlier in this loop)
                new_players_df = roster_df[~roster_df["PLAYER_ID"].isin(seen_player_ids)]

                if len(new_players_df) > 0:
                    new_players_df = new_players_df[["PLAYER_ID", "PLAYER"]].copy()
                    new_players_df.rename(columns={"PLAYER": "PLAYER_NAME"}, inplace=True)

                    all_players.extend(new_players_df.to_dict("records"))
                    seen_player_ids.update(new_players_df["PLAYER_ID"].tolist())

                    print(f"  ✓ Found {len(new_players_df)} new unique players")
                else:
                    print("  - No new players")

            except Exception as season_error:
                print(f"  ⚠ Error fetching roster: {str(season_error)}")
                continue

    except Exception as e:
        print(f"  ⚠ Error processing {team_name}: {str(e)}")

    print("-" * 40)

# --- FINAL UPDATE ---
if all_players:
    players_df = pd.DataFrame(all_players)
    players_df = players_df.drop_duplicates(subset=["PLAYER_ID"])
    players_df = normalize_for_postgres(players_df)

    print(f"\nPushing {len(players_df)} new players to database...")
    # CHANGED: 'append' instead of 'replace'
    players_df.to_sql("players", engine, if_exists="append", index=False)
    print("✅ Successfully appended new players.")
else:
    print("\n🙌 No new players found to add. Your 'players' table is already up to date!")

In [2]:
import time

import pandas as pd
from nba_api.stats.endpoints import commonplayerinfo
from tqdm import tqdm

# ----------------------------------
# INPUT PLAYER IDS
# ----------------------------------
PLAYER_IDS = [
    283,
    1729,
    1763,
    1903,
    2046,
    2049,
    2078,
    2366,
    2409,
    2416,
    2648,
    2668,
    2765,
    101146,
    101158,
    101215,
    200772,
    200781,
    200822,
    201169,
    201191,
    201286,
    201291,
    201629,
    201821,
    202068,
    202079,
    202197,
    202359,
    202385,
    202395,
    202458,
    202536,
    202545,
    202728,
    202775,
    202814,
    202862,
    202880,
    203130,
    203147,
    203183,
    203203,
    203474,
    203510,
    203548,
    203565,
    203590,
    203805,
    203942,
    203945,
    203951,
    203958,
    203966,
    203968,
    204021,
    204033,
    204040,
    204076,
    204079,
    204179,
    204222,
    1626205,
    1626208,
    1626214,
    1626218,
    1626260,
    1626262,
    1626266,
    1626296,
    1626643,
    1627293,
    1627760,
    1627791,
    1627822,
    1627861,
    1627866,
    1627879,
    1628238,
    1628419,
    1628435,
    1628450,
    1628451,
    1628454,
    1628473,
    1628475,
    1628492,
    1628504,
    1628506,
    1628515,
    1628591,
    1628605,
    1628610,
    1628769,
    1629005,
    1629037,
    1629044,
    1629093,
    1629150,
    1629152,
    1629155,
    1629232,
    1629309,
    1629460,
    1629600,
    1629602,
    1629606,
    1629620,
    1629668,
    1629689,
    1629742,
    1629751,
    1629755,
    1629756,
    1629783,
    1629788,
    1629873,
    1629958,
    1630196,
    1630211,
    1630223,
    1630257,
    1630266,
    1630278,
    1630283,
    1630285,
    1630286,
    1630306,
    1630542,
    1630555,
    1630562,
    1630565,
    1630585,
    1630605,
    1630607,
    1630608,
    1630610,
    1630622,
    1630640,
    1630686,
    1630693,
    1630698,
    1630701,
    1630707,
    1630758,
    1630762,
    1630787,
    1631113,
    1631160,
    1631167,
    1631301,
    1631320,
    1631376,
    1631386,
    1641777,
    1641789,
    1641793,
    1641806,
    1641811,
    1641851,
    1641857,
    1641877,
    1641879,
    1641907,
    1641926,
    1641970,
    1642385,
    1642422,
    1642439,
    1642486,
    1642502,
]

# ----------------------------------
# FETCH PLAYER INFO
# ----------------------------------
rows = []

for pid in tqdm(PLAYER_IDS, desc="Fetching player info"):
    try:
        info = commonplayerinfo.CommonPlayerInfo(
            player_id=pid,
            timeout=30,  # ✅ valid and supported
        )

        df = info.common_player_info.get_data_frame()

        if df.empty:
            continue

        row = df.iloc[0]

        rows.append(
            {
                "player_id": pid,
                "full_name": row["DISPLAY_FIRST_LAST"],
                "first_name": row["FIRST_NAME"],
                "last_name": row["LAST_NAME"],
                "is_active": row["ROSTERSTATUS"] == "Active",
                "birthdate": row["BIRTHDATE"],
                "country": row["COUNTRY"],
                "height": row["HEIGHT"],
                "weight": row["WEIGHT"],
                "position": row["POSITION"],
                "team_id": row["TEAM_ID"],
                "team_name": row["TEAM_NAME"],
                "team_abbreviation": row["TEAM_ABBREVIATION"],
                "from_year": row["FROM_YEAR"],
                "to_year": row["TO_YEAR"],
            }
        )

        time.sleep(0.6)  # rate-limit safety

    except Exception as e:
        rows.append({"player_id": pid, "error": str(e)})
        time.sleep(1)

players_df = pd.DataFrame(rows)

players_df.to_csv("player_metadata.csv", index=False)
players_df.to_json("player_metadata.json", orient="records")

print(f"Fetched {len(players_df)} players")

Fetching player info: 100%|██████████████████████████████████████████████████████████| 171/171 [02:25<00:00,  1.18it/s]

Fetched 171 players
